# Séance 3 · Simuler les lois binomiale et de Poisson

Lancez toutes les cellules : **Exécuter tout** (*Run All*), en haut du
notebook. Si VS Code demande un noyau, prenez **Python Environments**, puis
**Python 3.12**, comme pour `00-verifier-environnement.ipynb`.

Les cellules marquées **À vous** attendent votre code ou votre réponse.
À la fin : enregistrez, puis commit et push.

Sauf mention d'une source, les chiffres de ce notebook sont fictifs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom, poisson

rng = np.random.default_rng(2026)   # la graine : changez-la, les résultats bougent un peu

## 1. Une réservation : une épreuve de Bernoulli

Une réservation est annulée avec la probabilité $p = 0{,}37$ : c'est la
fréquence des annulations observée dans le jeu Hotel booking demand
(N. Antonio, A. de Almeida, L. Nunes, *Data in Brief*, 22 (2019), 41–49,
doi:10.1016/j.dib.2018.11.126, licence CC BY 4.0).

`rng.random()` tire un nombre au hasard entre 0 et 1. Il est inférieur à
$p$ avec la probabilité $p$ : c'est une épreuve de Bernoulli.

In [ ]:
p = 0.37
annulee = rng.random() < p
print("Réservation annulée :", annulee)

dix = rng.random(10) < p
print("Dix réservations :", dix.astype(int))
print("Nombre d'annulations :", dix.sum())

## 2. Vingt réservations : la loi binomiale

On simule 10 000 séries de 20 réservations. Dans chaque série, on compte les
annulations : c'est une valeur de $X$, qui suit la loi binomiale
$B(20 ; 0{,}37)$. On compare ensuite la fréquence de chaque valeur à la
probabilité donnée par la formule.

In [ ]:
n, series = 20, 10_000
reservations = rng.random((series, n)) < p     # un tableau de 10 000 lignes et 20 colonnes
X = reservations.sum(axis=1)                   # le nombre d'annulations de chaque série

valeurs = np.arange(n + 1)
frequences = np.bincount(X, minlength=n + 1) / series

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(valeurs, frequences, color="lightgrey", label="fréquence simulée")
ax.plot(valeurs, binom.pmf(valeurs, n, p), "o", color="black", label="probabilité, loi B(20 ; 0,37)")
ax.set_xlabel("nombre d'annulations parmi 20 réservations")
ax.set_ylabel("fréquence ou probabilité")
ax.legend()
plt.show()

In [ ]:
print("Moyenne simulée  :", round(X.mean(), 3), "   espérance n p        :", n * p)
print("Variance simulée :", round(X.var(), 3), "   variance n p (1 - p) :", round(n * p * (1 - p), 3))

### À vous

1. Estimez par la simulation la probabilité d'avoir **au moins 10
   annulations** sur 20 réservations : `np.mean(X >= 10)`. Comparez avec la
   valeur exacte, `binom.sf(9, n, p)`.
2. L'hôtel de villégiature du même jeu de données annule 27,8 % de ses
   réservations. Pour un week-end de 200 réservations, calculez l'espérance
   et l'écart-type du nombre d'annulations, puis vérifiez-les par une
   simulation.

In [ ]:
# À vous : question 1

In [ ]:
# À vous : question 2

## 3. Les arrivées à la réception : la loi de Poisson

Entre 17 h et 18 h, un hôtel reçoit en moyenne $\lambda = 6$ clients à la
réception. On simule 10 000 de ces heures avec `rng.poisson`, puis on
compare comme plus haut.

In [ ]:
lam, heures = 6, 10_000
arrivees = rng.poisson(lam, heures)

valeurs = np.arange(arrivees.max() + 1)
frequences = np.bincount(arrivees) / heures

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(valeurs, frequences, color="lightgrey", label="fréquence simulée")
ax.plot(valeurs, poisson.pmf(valeurs, lam), "o", color="black", label="probabilité, loi de Poisson de paramètre 6")
ax.set_xlabel("nombre d'arrivées en une heure")
ax.set_ylabel("fréquence ou probabilité")
ax.legend()
plt.show()

print("Moyenne simulée  :", round(arrivees.mean(), 3), "   espérance λ :", lam)
print("Variance simulée :", round(arrivees.var(), 3), "   variance λ  :", lam)

### À vous

3. Quelle est la probabilité de recevoir **plus de 10 clients** en une heure ?
   Estimez-la par la simulation, puis calculez-la avec `poisson.sf(10, lam)`.
4. La direction veut connaître le nombre d'arrivées qui n'est dépassé
   qu'une heure sur vingt. Trouvez le plus petit $k$ tel que
   $P(X \le k) \ge 0{,}95$ : essayez `poisson.cdf(k, lam)` pour plusieurs
   valeurs de $k$, ou `poisson.ppf(0.95, lam)`.

In [ ]:
# À vous : questions 3 et 4

## 4. De la binomiale à la loi de Poisson

Un hôtel de 400 chambres. Chaque nuit, chaque chambre signale une panne avec
la probabilité 0,005, indépendamment des autres. Le nombre de pannes suit la
loi binomiale $B(400 ; 0{,}005)$, de moyenne $400 \times 0{,}005 = 2$. La loi
de Poisson de paramètre 2 en est très proche : beaucoup d'épreuves, un
succès rare.

In [ ]:
k = np.arange(9)
print(" k   binomiale   Poisson")
for valeur, b, q in zip(k, binom.pmf(k, 400, 0.005), poisson.pmf(k, 2)):
    print(f"{valeur:2d}   {b:.4f}      {q:.4f}")

### À vous

5. Refaites la comparaison avec 40 chambres et une probabilité de 0,05 : la
   moyenne vaut toujours 2. Les deux lois sont-elles aussi proches ?
   Pourquoi ?

In [ ]:
# À vous : question 5

## Pour finir

Enregistrez le notebook, puis, dans **Contrôle de code source** : un
message comme « Séance 3 : notebook de simulation », **Valider**, puis
**Synchroniser les modifications**.

*Notebook du cours MTH101 (Adrien Chiodo, 2026), versé dans le domaine
public : [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/deed.fr).
Copiez-le, modifiez-le librement.*